# Discrete Speech Resynthesis and Speech Continuation walk-through

Below we will see how to use textless-lib to resynthesis speech and generate speech continuations.

In [1]:
!python --version

Python 3.8.20


In [2]:
!pip --version

pip 23.3.1 from /home/an08017/.conda/envs/burushaskitoenglish/lib/python3.8/site-packages/pip (python 3.8)


### Prerequisites

We'll need fairseq, textless and a bit of other dependencies.

At the moment there is a caveat that Colab doesn't support numpy versions above 1.21, which are required for some textless-lib functionality.

Here we'll use a small workaround by cloning and using a disk copy of textless. If you're running it locally, you can just uncomment the next line

In [3]:
! pip install git+https://github.com/pytorch/fairseq.git@dd106d9534b22e7db859a6b87ffd7780c38341f8 # need pip <24.1
! git clone https://github.com/facebookresearch/textlesslib.git && cd textless && pip install -e .

  Cloning https://github.com/pytorch/fairseq.git (to revision dd106d9534b22e7db859a6b87ffd7780c38341f8) to /tmp/pip-req-build-338anw3b
  Running command git clone --filter=blob:none --quiet https://github.com/pytorch/fairseq.git /tmp/pip-req-build-338anw3b
  Running command git rev-parse -q --verify 'sha^dd106d9534b22e7db859a6b87ffd7780c38341f8'
  Running command git fetch -q https://github.com/pytorch/fairseq.git dd106d9534b22e7db859a6b87ffd7780c38341f8
  Running command git checkout -q dd106d9534b22e7db859a6b87ffd7780c38341f8
  Resolved https://github.com/pytorch/fairseq.git to commit dd106d9534b22e7db859a6b87ffd7780c38341f8
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour chang

In [4]:
! pip install numpy==1.23.5         # Original paper uses 1.21.5

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# The following warning is showing up, for now I don't see any problems. If it works don't change it!
# DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. 
# A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. 
# Discussion can be found at https://github.com/pypa/pip/issues/12063

! pip install torch>=1.1.0 
! pip install torchaudio 
! pip install AMFM_decompy 
! pip install librosa==0.9.2        # PROBLEM VCODER - SOLVED use 0.9.2
! pip install scikit-learn
! pip install threadpoolctl==3.0.0  # PROBLEM - scikit-learn 1.7.2 requires threadpoolctl>=3.1.0, but you have threadpoolctl 3.0.0 which is incompatible. - SOLVED use numpy 1.23.5
! pip install numba==0.53.0         # PROBLEM - 0.53.0 Requires-Python >=3.6,<3.10 - SOLVEDusing python 8
! pip install joblib  
! pip install unidecode
! pip install inflect

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
DEPRECATION: omegaconf 2.0.6 has a non

In [6]:
! pip install git+https://github.com/pytorch/fairseq.git@dd106d9534b22e7db859a6b87ffd7780c38341f8
! git clone https://github.com/facebookresearch/textlesslib.git

  Cloning https://github.com/pytorch/fairseq.git (to revision dd106d9534b22e7db859a6b87ffd7780c38341f8) to /tmp/pip-req-build-m1jm82jq
  Running command git clone --filter=blob:none --quiet https://github.com/pytorch/fairseq.git /tmp/pip-req-build-m1jm82jq
  Running command git rev-parse -q --verify 'sha^dd106d9534b22e7db859a6b87ffd7780c38341f8'
  Running command git fetch -q https://github.com/pytorch/fairseq.git dd106d9534b22e7db859a6b87ffd7780c38341f8
  Running command git checkout -q dd106d9534b22e7db859a6b87ffd7780c38341f8
  Resolved https://github.com/pytorch/fairseq.git to commit dd106d9534b22e7db859a6b87ffd7780c38341f8
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour chang

In [7]:
cd textlesslib

/home/an08017/Documents/FYP-YAARAN/textlesslib


In [8]:
import textless

In [9]:
import IPython.display as ipd

import torchaudio, os, sys
import torch
import pathlib

import textless
from textless.data.speech_encoder import SpeechEncoder
# from textless.data.quantized_datasets import QuantizedLibriSpeech     # NEEDED ONLY FOR LIBRI SPEECH, we are not using LIBRI SPEECH data
# from textless.vocoders.tacotron2.vocoder import TacotronVocoder       # REQUIRES NVIDIA
from textless.vocoders.hifigan.vocoder import CodeHiFiGANVocoder        # CAN USE CPU
from textless.checkpoint_manager import CHECKPOINT_MANAGER

In [10]:
# Models available 
# (since we are using CPU now we are limited to using hifi-gan vcoder, 
# which limits us to what encoder and other parameters we can choose)
for key in CHECKPOINT_MANAGER.storage.keys():
    print(key)

hubert-base-ls960
mhubert-base-vp_en_es_fr
mhubert-base-vp_mls_cv_8lang
mhubert-base-25hz
hubert-base-ls960-kmeans-50
hubert-base-ls960-kmeans-100
hubert-base-ls960-kmeans-200
hubert-base-ls960-kmeans-500
mhubert-base-vp_en_es_fr-layer-11-kmeans-1000
hubert-base-ls960-layer-9-kmeans-500
hubert-base-ls960-layer-9-kmeans-expresso-2000
mhubert-base-vp_mls_cv_8lang-kmeans-2000
mhubert-base-vp_mls_cv_8lang-kmeans-expresso-2000
mhubert-base-25hz-kmeans-500
hubert-base-ls960-kmeans-50-tacotron
hubert-base-ls960-kmeans-100-tacotron
hubert-base-ls960-kmeans-200-tacotron
hubert-base-ls960-kmeans-50-tacotron-codes
hubert-base-ls960-kmeans-100-tacotron-codes
hubert-base-ls960-kmeans-200-tacotron-codes
mhubert-base-25hz-kmeans-500-hifigan
mhubert-base-25hz-kmeans-500-hifigan-config
hubert-base-ls960-layer-9-kmeans-500-hifigan
hubert-base-ls960-layer-9-kmeans-500-hifigan-config
hubert-base-ls960-layer-9-kmeans-500-hifigan-speakers
hubert-base-ls960-layer-9-kmeans-500-hifigan-styles
hubert-base-ls960

## Resynthesis

Firstly, let us configure what dense model and quantizer we will use:

In [22]:
dense_model_name = "hubert-base-ls960-layer-9"
quantizer_name = "kmeans"
vocab_size = 500 # one of [50, 100, 200]


We can initialise a SpeechEncoder using its name; this way a corresponding checkpoint will be downloaded automatically

In [23]:
encoder = SpeechEncoder.by_name(
    dense_model_name=dense_model_name,
    quantizer_model_name=quantizer_name,
    vocab_size=vocab_size,
    need_f0=False,
    deduplicate=True,
    f0_normalizer=None,
    f0_quantizer=None,
).to("cpu")

We will use a LibriSpeech dataset for our example. We can start with a vanilla version of it, load a single example and listen to it:

! mkdir -p datasets

raw_dataset = torchaudio.datasets.LIBRISPEECH(
    root="./datasets",
    url="dev-clean",
    download=True,
)

In our case we will use the bsk dataset from mozilla

In [7]:
!pip install librosa

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [45]:
# import librosa
# import torch
# import os

# dataset_path = "/home/an08017/Documents/FYP-YAARAN/mozilla_dataset/cv-corpus-23.0-2025-09-05/bsk/clips" # burushaski clips available on the mozilla link

# files = [f for f in os.listdir(dataset_path) if f.endswith((".mp3"))]

# waveforms = []
# for f in files:
#     filepath = os.path.join(dataset_path, f)
    
#     # Use librosa instead of torchaudio for MP3
#     waveform_np, sample_rate = librosa.load(filepath, sr=None, mono=False)
    
#     # Convert numpy to torch tensor with correct shape (channels, samples)
#     if waveform_np.ndim == 1:
#         waveform = torch.from_numpy(waveform_np).unsqueeze(0)  # Add channel dimension
#     else:
#         waveform = torch.from_numpy(waveform_np)
    
#     waveforms.append((waveform, sample_rate, filepath))
#     break

# print(len(waveforms))
import os
import torch
import torchaudio
from pydub import AudioSegment
from tqdm import tqdm

dataset_path = "/home/an08017/Documents/FYP-YAARAN/mozilla_dataset/cv-corpus-23.0-2025-09-05/bsk/clips"
output_path = "/home/an08017/Documents/FYP-YAARAN/mozilla_dataset/cv-corpus-23.0-2025-09-05/bsk/clips_wav"

os.makedirs(output_path, exist_ok=True)

files = [f for f in os.listdir(dataset_path) if f.endswith((".mp3"))]

waveforms = []
for f in files:
    mp3_filepath = os.path.join(dataset_path, f)
    wav_filename = f.replace('.mp3', '.wav')
    wav_filepath = os.path.join(output_path, wav_filename)
    
    if not os.path.exists(wav_filepath):
        audio = AudioSegment.from_mp3(mp3_filepath)
        
        if audio.channels > 1:
            audio = audio.set_channels(1)
        
        if audio.frame_rate != 16000:
            audio = audio.set_frame_rate(16000)
        
        audio.export(wav_filepath, format="wav")
    
    waveform, sample_rate = torchaudio.load(wav_filepath)
    waveforms.append((waveform, sample_rate, wav_filepath))

print(len(waveforms))

9301


In [19]:
# dataset_path = "/home/an08017/Documents/FYP-YAARAN/mozilla_dataset/cv-corpus-23.0-2025-09-05/bsk/clips" # burushaski clips available on the mozila link

# files = [f for f in os.listdir(dataset_path) if f.endswith((".mp3"))]

# waveforms = []
# for f in files:
#     filepath = os.path.join(dataset_path, f)
#     waveform, sample_rate = torchaudio.load(filepath)
#     waveforms.append((waveform, sample_rate, filepath))
#     break

# print(len(waveforms))


RuntimeError: Failed to load audio from /home/an08017/Documents/FYP-YAARAN/mozilla_dataset/cv-corpus-23.0-2025-09-05/bsk/clips/common_voice_bsk_41999914.mp3

In [46]:
# Make sure waveforms loaded successfully
if len(waveforms) > 0:
    audio, input_sample_rate, *_ = waveforms[0]
    print(f"✓ Audio loaded: shape {audio.shape}, sample rate {input_sample_rate}Hz")
    audio
else:
    print("✗ No audio loaded! Check the previous cell.")

✓ Audio loaded: shape torch.Size([1, 39168]), sample rate 16000Hz


In [20]:
# audio, input_sample_rate, *_ = waveforms[0]
# audio

IndexError: list index out of range

In [47]:
import IPython.display as ipd

if len(waveforms) > 0:
    ipd.Audio(audio, rate=input_sample_rate)
else:
    print("Cannot play audio - waveforms is empty")

In [ ]:
# ipd.Audio(audio, rate=input_sample_rate)

We can encode this audio example using our SpeechEncoder. The encoded audio is represented as a dictionary with key-value pairs:

In [48]:
encoded_audio = encoder(audio)
encoded_audio.keys()

dict_keys(['units', 'durations', 'dense'])

'units' contains the pseudo-unit stream, while 'durations' encodes per-token durations and 'dense' returns the original HuBERT representation of the audio.

Let's have a look how units look:

In [49]:
encoded_audio['units'][:10]

tensor([392, 296,  17, 296, 317, 249, 317, 461,  20, 461], dtype=torch.int32)

In [50]:
encoded_audio['durations'][:10]

tensor([1, 1, 2, 3, 2, 1, 2, 2, 1, 1])

Alternatively, textless-lib provides a simple wrapper around it which will return a "textless" representation of datapoints:
dataset = QuantizedLibriSpeech(
    encoder,
    root="./datasets",
    url="dev-clean",
    download=False,
)

datum = dataset[7]
datum['units'][:10]

Now let us initialise a corresponding Tacotron instance with a matching configuration:

In [34]:
# This requires NVIDIA 
# vocoder = TacotronVocoder.by_name(
#     dense_model_name,
#     quantizer_name,
#     vocab_size,
# ).to("cpu")

In [51]:
vocoder = CodeHiFiGANVocoder.by_name(
    dense_model_name = dense_model_name,
    quantizer_model_name=quantizer_name,
    vocab_size = vocab_size,
).to("cpu")

Removing weight norm...
CodeHiFiGAN model loaded!


Everythins is ready for resynthesising the unit stream back into the audio:

In [52]:
resynth_audio = vocoder(encoded_audio['units'])

In [53]:
ipd.Audio(resynth_audio.cpu().numpy(), rate=vocoder.output_sample_rate)

We haven't even started with the tasks below!

## Speech Continuation

One additional component we need is a unit-level language model. Here we re-use on from the GSLM paper

In [54]:
sys.path.append(str(pathlib.Path(textless.__path__[0]).parent / 'examples' / 'gslm/'))
from sampler import UnitLanguageModelSampler

...and download a pre-trained checkpoint (more checkpoints [here](https://github.com/pytorch/fairseq/tree/main/examples/textless_nlp/gslm/ulm)).

In [55]:
#downloading with Python instead of wget
import os
import requests
import tarfile

os.makedirs("LM", exist_ok=True)

url = "https://dl.fbaipublicfiles.com/textless_nlp/gslm/hubert/lm_km200/hubert200_lm.tgz"
output = "LM/hubert200_lm.tgz"

print("Downloading...")
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
response.raise_for_status()

with open(output, 'wb') as f:
    f.write(response.content)

print("Extracting...")
with tarfile.open(output, 'r:gz') as tar:
    tar.extractall('LM/')

print("✓ Done!")

Downloading...
Extracting...
✓ Done!


In [35]:
# ! mkdir -p LM && \
#     wget https://dl.fbaipublicfiles.com/textless_nlp/gslm/hubert/lm_km200/hubert500_lm.tgz -O LM/hubert500_lm.tgz && \
#     cd LM/ && \
#     tar -xvf hubert500_lm.tgz

--2025-12-03 10:47:17--  https://dl.fbaipublicfiles.com/textless_nlp/gslm/hubert/lm_km200/hubert500_lm.tgz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.161.104.79, 3.161.104.22, 3.161.104.108, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.161.104.79|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2025-12-03 10:47:18 ERROR 403: Forbidden.



We take the first 5 seconds of the same audio as a prompt:

In [56]:
prompt = audio[:, :input_sample_rate * 5]
ipd.Audio(prompt, rate=input_sample_rate)

...and encode it into the unit stream and double-check how the resynthesised version sounds like:

In [57]:
encoded = encoder(prompt)
units = encoded['units']
units

tensor([392, 296,  17, 296, 317, 249, 317, 461,  20, 461,  20, 461, 140,  20,
        249,  82, 305, 435, 140, 461, 184,  80,  73, 289,  82, 320, 305,   7,
        345, 389, 274,  43,   8, 354, 345, 255, 425, 251, 241, 431, 443, 151,
        240, 348,  35, 280, 300, 242, 469, 313, 368, 251, 368, 453, 168, 180,
        443, 319, 203, 381, 117,  13, 414, 170, 421, 305, 237, 187,  75, 305,
        442,  82, 305, 442, 421,  20, 128,  20, 128, 193,  17],
       dtype=torch.int32)

In [58]:
resynth_prompt = vocoder(units).cpu().numpy()
ipd.Audio(resynth_prompt, rate=vocoder.output_sample_rate)

Now we load our downloaded checkpoint and define some parameters for the LM sampling:

In [59]:
sampler = UnitLanguageModelSampler.from_pretrained(model_name_or_path="LM/hubert200_lm")

In [60]:
sampling_kwargs = {
    "temperature": 0.7,
    "sampling": True,
    "beam": 1,
    "prefix_size": -1,
    "max_len_a": 0.0,
    "max_len_b": 400,
}

In [61]:
# It is a fairseq-based language model, so it accepts text strings as an input.
unit_str = " ".join(list(map(str, units.tolist())))
sampled_unit_str = sampler.sample([unit_str], **sampling_kwargs)[0]

# Filter out <unk> tokens before converting to int
continuation = torch.tensor([
    int(x) for x in sampled_unit_str.split() 
    if x.isdigit() and x != '<unk>'
]).cpu()

continuation

tensor([ 17,  20,  20, 140,  20,  82, 140, 184,  80,  73,  82,   7,  43,   8,
        151,  35, 168, 180, 117,  13, 170, 187,  75,  82,  20, 128,  20, 128,
        193,  17,  93,  26,  26, 165, 101,   9,  70,   7, 142,  26,  49, 184,
         32,  36,  89,  72,  25,  26, 188, 152, 162, 129, 188,  90,   2,   8,
         22, 117,  22, 122, 110,  18,  35, 114, 127, 113, 115, 172, 175,  43,
         91, 164, 173,  65,   6, 121,  59,  56,  25,  82,  63, 182, 177, 106,
        170,  82, 114, 194, 198,  12,  13, 156,  44, 144, 198,  12,  87, 157,
        144, 198, 123, 192,  87,  39, 152,  36, 133,   8, 133,   8, 117,   8,
         22, 117,  10, 122,  96,  12,  95,  85,  34, 196, 150,  81,  52,  97,
        108, 184, 123, 185, 129,  68,  36, 162, 166, 191,  11, 117,  22, 117,
         10, 130, 128, 145,  97, 145, 108, 184,  12,  40,  67,  98, 187, 172,
        185, 156, 175,  43,  91, 173, 128, 116, 193, 170,  79, 104, 136, 115,
         24,  40,  67,  98,  91,  65, 172,  71,  31, 100,  37, 1

In [42]:
# # It is a fairseq-based language model, so it accepts text strings as an input.
# unit_str = " ".join(list(map(str, units.tolist())))
# sampled_unit_str = sampler.sample([unit_str], **sampling_kwargs)[0]
# continuation = torch.tensor([int(x) for x in sampled_unit_str.split()]).cuda()
# continuation

ValueError: invalid literal for int() with base 10: '<unk>'

Now, given this unit-level continuation we can vocode it into the audio:

In [62]:
resynth_continuation = vocoder(continuation).cpu().numpy()[:10 * vocoder.output_sample_rate]
ipd.Audio(resynth_continuation, rate=vocoder.output_sample_rate)